# Crop Yield Prediction Under Changing Climate Conditions

**Dataset:** Crop Yield Prediction Dataset (Kaggle, patelris) — merged from FAO (yield, pesticide use) and World Bank (rainfall, temperature).  
**Scope:** 28,242 records | 101 countries | 10 staple crops | 1990–2013

This notebook provides an end-to-end Machine Learning and Climate Analytics workflow:
1. **Data Ingestion & Hygiene:** Cleaning, unit normalization ($hg/ha \rightarrow t/ha$), and deduplication.
2. **Exploratory Data Analysis (EDA):** Yield trajectory over time, distribution across crop families, climate correlations, and crop-specific temperature dynamics.
3. **Temporal Train/Test Split:** Realistic backtesting by training on $\le 2009$ and evaluating on unseen future years ($2010–2013$).
4. **Benchmarking Regression Algorithms:** Linear Regression, Ridge, Random Forest Regressor, and Gradient Boosting.
5. **Feature Importance Interpretability:** Unpacking the structural dominance of geography/crop versus climate variables.
6. **Lightweight Browser Model Export:** Exporting tree nodes to JSON for zero-server client-side interactive inference.

In [ ]:
import json
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import joblib

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

DATA_PATH = '../data/yield_df.csv'
FIG_DIR = '../figures'
MODEL_DIR = '../models'
EXPORT_DIR = '../exports'

for d in [FIG_DIR, MODEL_DIR, EXPORT_DIR]:
    os.makedirs(d, exist_ok=True)

## 1. Load & Clean Data

- Rename fields to standard snake_case.
- Convert yield from hectograms per hectare ($hg/ha$) to standard agricultural metric tonnes per hectare ($t/ha$).
- Deduplicate identical country-crop-year entries.

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
df = df.rename(columns={
    'Area': 'country',
    'Item': 'crop',
    'Year': 'year',
    'hg/ha_yield': 'yield_hg_ha',
    'average_rain_fall_mm_per_year': 'rainfall_mm',
    'pesticides_tonnes': 'pesticides_tonnes',
    'avg_temp': 'avg_temp_c',
})
df['yield_tons_ha'] = df['yield_hg_ha'] / 10000  # hectograms/ha -> tonnes/ha
df = df.drop_duplicates()

print(f'Final cleaned shape: {df.shape}')
print('Null values per column:\n', df.isnull().sum())
print('\nTracked crops:\n', df['crop'].unique())
df.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Global yield trend over time
yearly = df.groupby('year')['yield_tons_ha'].mean().reset_index()
plt.figure(figsize=(8, 4))
plt.plot(yearly['year'], yearly['yield_tons_ha'], marker='o', color='#3B6B4A', linewidth=2.5)
plt.title('Global Average Crop Yield (1990–2013)', fontsize=13, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Yield (tonnes/ha)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/yield_trend.png')
plt.show()

In [ ]:
# Yield by crop
crop_order = df.groupby('crop')['yield_tons_ha'].mean().sort_values(ascending=False).index
plt.figure(figsize=(9, 4.5))
sns.boxplot(data=df, x='crop', y='yield_tons_ha', order=crop_order, color='#C08A2E', fliersize=2)
plt.xticks(rotation=35, ha='right')
plt.title('Yield Distribution by Crop Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/yield_by_crop.png')
plt.show()

In [ ]:
# Climate correlations
plt.figure(figsize=(6, 5))
corr = df[['yield_tons_ha', 'rainfall_mm', 'pesticides_tonnes', 'avg_temp_c', 'year']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0)
plt.title('Correlation Heatmap', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/correlation_heatmap.png')
plt.show()

In [ ]:
# Temperature vs yield faceted by major crops
major_crops = ['Maize', 'Wheat', 'Rice, paddy', 'Soybeans']
plt.figure(figsize=(9, 7))
for i, crop in enumerate(major_crops, 1):
    plt.subplot(2, 2, i)
    sub = df[df['crop'] == crop]
    plt.scatter(sub['avg_temp_c'], sub['yield_tons_ha'], alpha=0.35, s=12, color='#3B6B4A')
    plt.title(crop, fontweight='bold')
    plt.xlabel('Avg Temp (°C)')
    plt.ylabel('Yield (t/ha)')
plt.suptitle('Temperature vs Yield Across Primary Global Grains', y=0.98, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/temp_vs_yield_by_crop.png')
plt.show()

## 3. Feature Engineering & Temporal Split

To simulate a genuine real-world forecasting task, we perform an **out-of-time temporal split**:
- **Training set:** 1990–2009 (21,305 records)
- **Test holdout:** 2010–2013 (4,627 records)

In [ ]:
FEATURES_NUM = ['year', 'rainfall_mm', 'pesticides_tonnes', 'avg_temp_c']
FEATURES_CAT = ['country', 'crop']
TARGET = 'yield_tons_ha'

train_df = df[df['year'] <= 2009].copy()
test_df = df[df['year'] > 2009].copy()
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

X_train, y_train = train_df[FEATURES_NUM + FEATURES_CAT], train_df[TARGET]
X_test, y_test = test_df[FEATURES_NUM + FEATURES_CAT], test_df[TARGET]

preprocess = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), FEATURES_CAT),
])

## 4. Train & Benchmark Regression Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Random Forest': RandomForestRegressor(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42
    ),
}

results = []
fitted = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    results.append({'model': name, 'r2': round(r2, 4), 'rmse': round(rmse, 3), 'mae': round(mae, 3)})
    fitted[name] = pipe
    print(f'{name:20s}  R2={r2:.4f}  RMSE={rmse:.3f} t/ha  MAE={mae:.3f} t/ha')

results_df = pd.DataFrame(results).sort_values('r2', ascending=False)
results_df.to_csv(f'{EXPORT_DIR}/model_comparison.csv', index=False)
display(results_df)

best_name = results_df.iloc[0]['model']
best_model = fitted[best_name]
print(f'\nBest model: {best_name}')

## 5. Feature Importance Analysis

In [ ]:
rf_pipe = fitted['Random Forest']
ohe_cols = rf_pipe.named_steps['prep'].named_transformers_['cat'].get_feature_names_out(FEATURES_CAT)
all_cols = FEATURES_NUM + list(ohe_cols)
importances = rf_pipe.named_steps['model'].feature_importances_

imp_df = pd.DataFrame({'feature': all_cols, 'importance': importances})
imp_df['group'] = imp_df['feature'].apply(
    lambda f: 'country' if f.startswith('country_') else ('crop' if f.startswith('crop_') else f)
)
grouped_imp = imp_df.groupby('group')['importance'].sum().sort_values(ascending=False)
print(grouped_imp)

plt.figure(figsize=(7, 4))
grouped_imp.plot(kind='barh', color='#C08A2E', edgecolor='#212B1E')
plt.gca().invert_yaxis()
plt.title('Random Forest Feature Importance (Grouped)', fontsize=12, fontweight='bold')
plt.xlabel('Cumulative Relative Importance')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/feature_importance.png')
plt.show()

## 6. Persist Production Model

In [ ]:
joblib.dump(best_model, f'{MODEL_DIR}/best_model.joblib')
print(f'Successfully serialized production model ({best_name}) to {MODEL_DIR}/best_model.joblib')

## 7. Export Client-Side Lightweight Random Forest Model (Browser Inference)

To deliver a real-time web dashboard without requiring a continuous Python server, we train a 25-tree Random Forest with the top 20 production countries + 'Other', and export recursive tree decision nodes directly to JSON.

In [ ]:
top_countries = df['country'].value_counts().head(20).index.tolist()
df['country_bucketed'] = df['country'].where(df['country'].isin(top_countries), 'Other')

FEATURES_LITE_NUM = ['year', 'rainfall_mm', 'pesticides_tonnes', 'avg_temp_c']
FEATURES_LITE_CAT = ['crop', 'country_bucketed']

X_lite = df[FEATURES_LITE_NUM + FEATURES_LITE_CAT]
y_lite = df[TARGET]

lite_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_lite_cat = lite_ohe.fit_transform(X_lite[FEATURES_LITE_CAT])
cat_names = lite_ohe.get_feature_names_out(FEATURES_LITE_CAT)
X_lite_full = np.hstack([X_lite[FEATURES_LITE_NUM].values, X_lite_cat])
feature_names_lite = FEATURES_LITE_NUM + list(cat_names)

lite_rf = RandomForestRegressor(n_estimators=25, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1)
lite_rf.fit(X_lite_full, y_lite)
lite_r2 = r2_score(y_lite, lite_rf.predict(X_lite_full))
print(f'Lightweight browser model in-sample R²: {lite_r2:.3f}')


def export_tree(tree):
    t = tree.tree_

    def node_to_dict(i):
        if t.children_left[i] == t.children_right[i] == -1:
            return {'leaf': round(float(t.value[i][0][0]), 4)}
        return {
            'f': int(t.feature[i]),
            't': round(float(t.threshold[i]), 5),
            'l': node_to_dict(t.children_left[i]),
            'r': node_to_dict(t.children_right[i]),
        }

    return node_to_dict(0)


forest_json = {
    'feature_names': feature_names_lite,
    'num_features': FEATURES_LITE_NUM,
    'crop_options': sorted(df['crop'].unique().tolist()),
    'country_options': sorted(top_countries) + ['Other'],
    'trees': [export_tree(t) for t in lite_rf.estimators_],
    'feature_ranges': {
        'year': [1990, 2030],
        'rainfall_mm': [int(df['rainfall_mm'].quantile(0.02)), int(df['rainfall_mm'].quantile(0.98))],
        'pesticides_tonnes': [0, int(df['pesticides_tonnes'].quantile(0.95))],
        'avg_temp_c': [float(round(df['avg_temp_c'].min(), 2)), float(round(df['avg_temp_c'].max(), 2))],
    },
}

with open(f'{EXPORT_DIR}/forest_model.json', 'w') as f:
    json.dump(forest_json, f)

print(f'Exported {len(forest_json["trees"])} trees to {EXPORT_DIR}/forest_model.json')

## 8. Export Aggregated Stats for Dashboard Visualizations

In [ ]:
dashboard_data = {
    'yearly_global_yield': yearly.round(3).to_dict(orient='records'),
    'yield_by_crop': (
        df.groupby('crop')['yield_tons_ha'].agg(['mean', 'median', 'std']).round(3)
        .reset_index().to_dict(orient='records')
    ),
    'top_countries_by_yield': (
        df.groupby('country')['yield_tons_ha'].mean().sort_values(ascending=False).head(12)
        .round(3).reset_index().to_dict(orient='records')
    ),
    'bottom_countries_by_yield': (
        df.groupby('country')['yield_tons_ha'].mean().sort_values().head(12)
        .round(3).reset_index().to_dict(orient='records')
    ),
    'temp_vs_yield_by_crop': {
        crop: df[df['crop'] == crop][['avg_temp_c', 'yield_tons_ha']]
        .sample(min(150, (df['crop'] == crop).sum()), random_state=1)
        .round(3).to_dict(orient='records')
        for crop in major_crops
    },
    'correlation': corr.round(3).to_dict(),
    'model_comparison': results_df.to_dict(orient='records'),
    'feature_importance': grouped_imp.round(4).reset_index().rename(columns={'index': 'group'}).to_dict(orient='records'),
    'dataset_stats': {
        'rows': int(len(df)),
        'countries': int(df['country'].nunique()),
        'crops': int(df['crop'].nunique()),
        'year_min': int(df['year'].min()),
        'year_max': int(df['year'].max()),
    },
}

with open(f'{EXPORT_DIR}/dashboard_data.json', 'w') as f:
    json.dump(dashboard_data, f)

print(f'Exported dashboard data to {EXPORT_DIR}/dashboard_data.json')
print('All processing complete!')